In [1]:
from functions import *

In [2]:
change_to_parent_dir_once()

base = Path("D:/OneDrive/文档/Synfiles/Project (LTR)/Dataset/almrrc2021/almrrc2021-data-training")

data = load_all_json_dicts(
    base_dir=base,
    output_dir="D:/OneDrive/文档/Synfiles/Project (LTR)/Dataset/almrrc2021/processed_json_pickles",
    rebuild=False
)
package_data = data["model_build_inputs__package_data"]
route_data = data["model_build_inputs__route_data"]
travel_times = data["model_build_inputs__travel_times"]
actual_sequences = data["model_build_inputs__actual_sequences"]

Loaded cached: model_apply_inputs__new_package_data, dict len=13
Loaded cached: model_apply_inputs__new_route_data, dict len=13
Loaded cached: model_apply_inputs__new_travel_times, dict len=13
Skipping empty JSON: model_apply_outputs\proposed_sequences.json
Loaded cached: model_build_inputs__actual_sequences, dict len=6112
Loaded cached: model_build_inputs__invalid_sequence_scores, dict len=6112
Loaded cached: model_build_inputs__package_data, dict len=6112
Loaded cached: model_build_inputs__route_data, dict len=6112
Loaded cached: model_build_inputs__travel_times, dict len=6112
Loaded cached: model_score_inputs__new_actual_sequences, dict len=13
Loaded cached: model_score_inputs__new_invalid_sequence_scores, dict len=13
Skipping empty JSON: model_score_outputs\scores.json
Skipping empty JSON: model_score_timings\model_apply_time.json
Skipping empty JSON: model_score_timings\model_build_time.json

Finished. Loaded 10 non-empty JSON dictionaries.


In [9]:
route_data_list = list(route_data.values())

In [11]:
route_data_list[0].keys()

dict_keys(['station_code', 'date_YYYY_MM_DD', 'departure_time_utc', 'executor_capacity_cm3', 'route_score', 'stops'])

In [3]:
route_id = "RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77"

# Get stops
stops = route_data[route_id]["stops"]

route_df = (
    pd.DataFrame.from_dict(stops, orient="index")
    .reset_index()
    .rename(columns={"index": "stop_id"})
)

# Get actual visiting sequence
seq_obj = actual_sequences[route_id]

# Handle common ALMRRC format
if isinstance(seq_obj, dict) and "actual" in seq_obj:
    sequence_map = seq_obj["actual"]
else:
    sequence_map = seq_obj

route_df["sequence"] = route_df["stop_id"].map(sequence_map)

# Sort by actual route order
route_ordered = (
    route_df
    .dropna(subset=["sequence"])
    .sort_values("sequence")
    .copy()
)

route_ordered["sequence"] = route_ordered["sequence"].astype(int)

In [26]:
route_data[route_id]

{'station_code': 'DLA3',
 'date_YYYY_MM_DD': '2018-07-27',
 'departure_time_utc': '16:02:10',
 'executor_capacity_cm3': 3313071.0,
 'route_score': 'High',
 'stops': {'AD': {'lat': 34.099611,
   'lng': -118.283062,
   'type': 'Dropoff',
   'zone_id': 'P-12.3C'},
  'AF': {'lat': 34.101587,
   'lng': -118.291125,
   'type': 'Dropoff',
   'zone_id': 'A-1.2D'},
  'AG': {'lat': 34.089727,
   'lng': -118.28553,
   'type': 'Dropoff',
   'zone_id': 'A-2.1A'},
  'BA': {'lat': 34.096132,
   'lng': -118.292869,
   'type': 'Dropoff',
   'zone_id': 'A-1.2C'},
  'BE': {'lat': 34.098482,
   'lng': -118.286243,
   'type': 'Dropoff',
   'zone_id': 'P-13.3B'},
  'BG': {'lat': 34.102251,
   'lng': -118.287403,
   'type': 'Dropoff',
   'zone_id': 'P-13.2A'},
  'BP': {'lat': 34.095585,
   'lng': -118.28179,
   'type': 'Dropoff',
   'zone_id': 'P-13.2C'},
  'BT': {'lat': 34.101474,
   'lng': -118.289609,
   'type': 'Dropoff',
   'zone_id': 'A-1.1D'},
  'BY': {'lat': 34.091721,
   'lng': -118.284539,
   'type

In [32]:
package_data[route_id]

{'AD': {'PackageID_9d7fdd03-f2cf-4c6f-9128-028258fc09ea': {'scan_status': 'DELIVERED',
   'time_window': {'start_time_utc': nan, 'end_time_utc': nan},
   'planned_service_time_seconds': 59.3,
   'dimensions': {'depth_cm': 25.4, 'height_cm': 7.6, 'width_cm': 17.8}},
  'PackageID_5541e679-b7bd-4992-b288-e862f6c84ae7': {'scan_status': 'DELIVERED',
   'time_window': {'start_time_utc': '2018-07-27 16:00:00',
    'end_time_utc': '2018-07-28 00:00:00'},
   'planned_service_time_seconds': 59.3,
   'dimensions': {'depth_cm': 25.4, 'height_cm': 12.7, 'width_cm': 17.8}},
  'PackageID_84d0295b-1adb-4a33-a65e-f7d6247c7a07': {'scan_status': 'DELIVERED',
   'time_window': {'start_time_utc': nan, 'end_time_utc': nan},
   'planned_service_time_seconds': 59.3,
   'dimensions': {'depth_cm': 39.4, 'height_cm': 7.6, 'width_cm': 31.8}}},
 'AF': {'PackageID_15c6a204-ec5f-4ced-9c3d-472316cc7759': {'scan_status': 'DELIVERED',
   'time_window': {'start_time_utc': '2018-07-27 16:00:00',
    'end_time_utc': '2018

In [14]:
actual_sequences[route_id]

{'actual': {'AD': 105,
  'AF': 47,
  'AG': 4,
  'BA': 33,
  'BE': 109,
  'BG': 53,
  'BP': 67,
  'BT': 49,
  'BY': 7,
  'BZ': 61,
  'CA': 43,
  'CG': 26,
  'CK': 86,
  'CM': 96,
  'CO': 54,
  'CP': 36,
  'CW': 8,
  'DJ': 28,
  'DL': 15,
  'DN': 37,
  'DQ': 35,
  'EC': 106,
  'EH': 101,
  'EO': 56,
  'EX': 104,
  'EY': 69,
  'FF': 88,
  'FH': 12,
  'FY': 76,
  'GB': 81,
  'GN': 42,
  'GP': 2,
  'GS': 64,
  'GU': 16,
  'GW': 74,
  'HB': 97,
  'HG': 87,
  'HN': 40,
  'HO': 80,
  'HR': 111,
  'HT': 3,
  'HW': 93,
  'IA': 57,
  'IJ': 20,
  'IM': 68,
  'IP': 94,
  'IW': 23,
  'JH': 10,
  'JM': 92,
  'KA': 48,
  'KG': 55,
  'KJ': 41,
  'KM': 51,
  'KN': 17,
  'KP': 22,
  'KU': 63,
  'LB': 11,
  'LD': 46,
  'LG': 27,
  'LK': 84,
  'LY': 83,
  'MA': 66,
  'MO': 89,
  'MQ': 70,
  'MR': 25,
  'MW': 50,
  'NE': 45,
  'NL': 29,
  'NM': 79,
  'NR': 18,
  'NU': 65,
  'PB': 110,
  'PJ': 73,
  'PS': 82,
  'PT': 14,
  'PX': 117,
  'QE': 85,
  'QM': 5,
  'QO': 107,
  'QX': 118,
  'RA': 71,
  'RG': 59,
  

In [34]:
travel_times['RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77']['AF']

{'AD': 209.8,
 'AF': 0.0,
 'AG': 348.3,
 'BA': 223.2,
 'BE': 219.2,
 'BG': 113.6,
 'BP': 342.2,
 'BT': 56.7,
 'BY': 351.9,
 'BZ': 280.0,
 'CA': 87.4,
 'CG': 231.8,
 'CK': 323.6,
 'CM': 428.1,
 'CO': 102.7,
 'CP': 284.4,
 'CW': 295.3,
 'DJ': 378.3,
 'DL': 273.5,
 'DN': 284.4,
 'DQ': 150.5,
 'EC': 227.9,
 'EH': 301.4,
 'EO': 173.2,
 'EX': 244.3,
 'EY': 319.2,
 'FF': 354.3,
 'FH': 281.9,
 'FY': 386.2,
 'GB': 317.2,
 'GN': 108.4,
 'GP': 388.9,
 'GS': 255.1,
 'GU': 210.1,
 'GW': 316.9,
 'HB': 437.5,
 'HG': 363.5,
 'HN': 229.5,
 'HO': 330.4,
 'HR': 182.4,
 'HT': 353.3,
 'HW': 374.5,
 'IA': 189.7,
 'IJ': 249.5,
 'IM': 351.0,
 'IP': 383.6,
 'IW': 220.0,
 'JH': 320.4,
 'JM': 448.4,
 'KA': 21.8,
 'KG': 104.3,
 'KJ': 127.5,
 'KM': 94.6,
 'KN': 256.4,
 'KP': 295.8,
 'KU': 284.0,
 'LB': 307.2,
 'LD': 94.7,
 'LG': 260.0,
 'LK': 348.4,
 'LY': 356.9,
 'MA': 336.2,
 'MO': 397.8,
 'MQ': 320.3,
 'MR': 292.6,
 'MW': 57.5,
 'NE': 161.9,
 'NL': 341.5,
 'NM': 369.2,
 'NR': 308.4,
 'NU': 262.1,
 'PB': 196.0,
